In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02_ingest_sources (config-driven)
# MAGIC **Source websites -> bronze Volume**, driven by `config/sources.yml`. No URL lives in this code.
# MAGIC
# MAGIC Per source, in order:
# MAGIC 1. **Resolve** the URL from the registry. For `latest_year` sources the newest published vintage is discovered by probing; for `static` sources the URL is used as-is.
# MAGIC 2. **Check the host** (allow-list on the configured URL *and* on the final URL after redirects).
# MAGIC 3. **Download** with retries, streaming to local disk while hashing.
# MAGIC 4. **Validate** the content (non-empty, expected header/columns, row counts) *before* it touches bronze.
# MAGIC 5. **Land** under a dated name in the Volume. Bronze is append-only, so history is kept.
# MAGIC 6. **Log every attempt** to `bronze_ingest_manifest`, including the vintage. Failures and blocked hosts are recorded, not hidden.
# MAGIC
# MAGIC **Parameters (widgets, so a Job can set them):** `config_path` (default `config/sources.yml`, relative to this notebook; run from a Git folder or point it at an uploaded copy), `sources` (`all` or a comma-separated list of names), and `census_api_key` (the ACS source needs a free Census API key; see below).
# MAGIC
# MAGIC **API keys:** `api.census.gov` answers a keyless request with an HTML "Missing Key" page under HTTP 200. Get a free key from Census (api.census.gov/data/key_signup.html), then either create a Databricks secret (scope/key named in `sources.yml`, preferred) or paste the key into the `census_api_key` widget. The key is added to the request at call time and scrubbed from the manifest, logs and error messages; it is never stored in the registry, the repo, or a URL.
# MAGIC
# MAGIC **Vintage discipline:** `latest_year` sources move forward on their own when a new vintage is published. The chosen vintage is written to the manifest every run. Set `pin:` in the registry to freeze it for a reproducible rebuild.

# COMMAND ----------

# --- CORE START ---
import csv
import hashlib
import json
import os
import re
import shutil
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import requests
import yaml
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def get_param(name, default):
    """Widget value if running in Databricks, else the default."""
    try:
        dbutils.widgets.text(name, default)
        return dbutils.widgets.get(name).strip() or default
    except NameError:
        return default


# ------------------------------------------------------------------ API keys
SECRETS = []   # values that must never appear in logs, the manifest, or error messages


def register_secret(value):
    if value and value not in SECRETS:
        SECRETS.append(value)


def scrub(text):
    """Remove any registered secret from text before it is printed or stored."""
    if text is None:
        return None
    out = str(text)
    for s in SECRETS:
        out = out.replace(s, "***")
    return out


def get_api_key(src):
    """Databricks secret first (preferred), then the notebook/job widget. Never read from the registry or the repo."""
    cfg = src.get("api_key")
    if not cfg:
        return None
    if cfg.get("secret_scope") and cfg.get("secret_key"):
        try:
            return dbutils.secrets.get(cfg["secret_scope"], cfg["secret_key"]) or None
        except Exception:
            pass   # scope not created: fall back to the widget
    return get_param(cfg["widget"], "") or None


def key_params_for(src):
    """Query-string params carrying the key. Added at request time only, so a keyed URL is never stored."""
    cfg = src.get("api_key")
    key = get_api_key(src) if cfg else None
    if not key:
        return None
    register_secret(key)
    return {cfg["query_param"]: key}


CATALOG = "bootcamp_students"
SCHEMA = "dataexpert_student_viknim"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/bronze_landing"
MANIFEST_TABLE = f"{CATALOG}.{SCHEMA}.bronze_ingest_manifest"

# Identify the project honestly; some CDNs reject the default python-requests agent.
USER_AGENT = "rental-analysis/1.0 (Databricks bootcamp project)"

FIPS_HEADER = "STATE|STATEFP|COUNTYFP|COUNTYNS|COUNTYNAME|CLASSFP|FUNCSTAT"
ZORI_REQUIRED_COLUMNS = {"RegionID", "RegionName", "StateCodeFIPS", "MunicipalCodeFIPS"}
ALLOWED_STRATEGIES = {"static", "latest_year", "manual"}


# ---------------------------------------------------------------- validators
# Each takes (local_path, src_config) and returns (ok, detail, file_vintage_or_None).
def validate_fips_reference(path, src):
    with open(path, encoding="utf-8") as f:
        lines = f.read().splitlines()
    if not lines:
        return False, "empty file", None
    if lines[0].strip() != FIPS_HEADER:
        return False, f"unexpected header: {lines[0][:200]!r}", None
    n = len(lines) - 1
    if n < 3000:
        return False, f"only {n} data rows (expected 3,000+)", None
    return True, f"{n} rows", None


def validate_zillow_zori(path, src):
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if not header:
            return False, "empty file", None
        missing = ZORI_REQUIRED_COLUMNS - set(header)
        if missing:
            return False, f"missing columns {sorted(missing)}; header starts {header[:12]}", None
        date_cols = [c for c in header if re.fullmatch(r"\d{4}-\d{2}-\d{2}", c)]
        if len(date_cols) < 12:
            return False, f"only {len(date_cols)} monthly columns", None
        n = sum(1 for _ in reader)
    if n < 500:
        return False, f"only {n} data rows", None
    latest = max(date_cols)
    return True, f"{n} rows, {len(date_cols)} monthly columns, latest {latest}", f"ZORI as of {latest}"


def acs_columns(src):
    """Estimate + margin-of-error column for every configured variable."""
    return [f"{v}{suffix}" for v in src["variables"] for suffix in ("E", "M")]


def validate_acs_api(path, src):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list) or len(data) < 2:
        return False, "not a header+rows JSON array", None
    header, rows = data[0], data[1:]
    needed = ["NAME", "state", "county"] + acs_columns(src)
    missing = [c for c in needed if c not in header]
    if missing:
        return False, f"missing columns {missing}", None
    s, c = header.index("state"), header.index("county")
    geoids = [r[s] + r[c] for r in rows]
    if len(set(geoids)) != len(geoids):
        return False, f"duplicate geoids: {len(geoids) - len(set(geoids))}", None
    if len(rows) < 3100:
        return False, f"only {len(rows)} rows (expected 3,100+)", None
    bad_len = [g for g in geoids if len(g) != 5]
    if bad_len:
        return False, f"{len(bad_len)} geoids not 5 chars, e.g. {bad_len[:3]}", None
    return True, f"{len(rows)} counties, {len(acs_columns(src))} columns", None


VALIDATORS = {
    "fips_reference": validate_fips_reference,
    "zillow_zori": validate_zillow_zori,
    "acs_api": validate_acs_api,
}


# ------------------------------------------------------------------ registry
def host_allowed(hostname, allowed_hosts):
    if not hostname:
        return False
    h = hostname.lower().rstrip(".")
    return any(h == a.lower() or h.endswith("." + a.lower()) for a in allowed_hosts)


def load_registry(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Source registry not found at {p.resolve()}. Run this notebook from a Git folder, "
            "or upload sources.yml and set the config_path widget."
        )
    with open(p, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return cfg["sources"]


def validate_registry(sources):
    """Fail at load time, with every problem listed, instead of halfway through a run."""
    errors, seen = [], set()
    for s in sources:
        n = s.get("name", "<missing name>")
        if n in seen:
            errors.append(f"{n}: duplicate name")
        seen.add(n)
        strat = s.get("strategy")
        if strat not in ALLOWED_STRATEGIES:
            errors.append(f"{n}: strategy must be one of {sorted(ALLOWED_STRATEGIES)}, got {strat!r}")
            continue
        for k in ("subdir", "stem", "ext"):
            if not s.get(k):
                errors.append(f"{n}: missing '{k}'")
        if strat == "manual":
            continue

        hosts = s.get("allowed_hosts")
        if not hosts:
            errors.append(f"{n}: missing 'allowed_hosts'")
        if s.get("validator") not in VALIDATORS:
            errors.append(f"{n}: validator {s.get('validator')!r} not in {sorted(VALIDATORS)}")
        if s.get("validator") == "acs_api" and not s.get("variables"):
            errors.append(f"{n}: acs_api validator needs a 'variables' list")
        if s.get("probe_expect") not in (None, "json"):
            errors.append(f"{n}: probe_expect must be 'json' or omitted, got {s.get('probe_expect')!r}")
        ak = s.get("api_key")
        if ak is not None:
            for k in ("query_param", "widget"):
                if not ak.get(k):
                    errors.append(f"{n}: api_key needs '{k}'")
            if bool(ak.get("secret_scope")) != bool(ak.get("secret_key")):
                errors.append(f"{n}: api_key needs both secret_scope and secret_key, or neither")
            qp = ak.get("query_param")
            if qp and any(f"{qp}=" in (s.get(k) or "") for k in ("url", "url_template", "probe_template")):
                errors.append(f"{n}: never put the key in a URL; the notebook adds '{qp}' at request time")

        url_keys = ["url"] if strat == "static" else ["url_template", "probe_template"]
        for k in url_keys:
            url = s.get(k)
            if not url:
                errors.append(f"{n}: missing '{k}'")
                continue
            u = urlparse(url)
            if u.scheme != "https":
                errors.append(f"{n}: '{k}' must be https")
            if hosts and not host_allowed(u.hostname, hosts):
                errors.append(f"{n}: host {u.hostname!r} in '{k}' is not in allowed_hosts {hosts}")

        if strat == "latest_year":
            yc = s.get("year_candidates") or {}
            if not (isinstance(yc.get("start_offset"), int) and isinstance(yc.get("count"), int) and yc["count"] > 0):
                errors.append(f"{n}: year_candidates needs integer start_offset and count > 0")
            if not s.get("vintage_label"):
                errors.append(f"{n}: missing 'vintage_label'")
    if errors:
        raise ValueError("Invalid source registry:\n  - " + "\n  - ".join(errors))


def format_url(template, src, year=None):
    variables = ",".join(acs_columns(src)) if src.get("variables") else ""
    return template.format(year=year, variables=variables)


# ------------------------------------------------------------- http + resolve
def make_session():
    s = requests.Session()
    retry = Retry(
        total=4, connect=4, read=4, backoff_factor=2,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "HEAD"), respect_retry_after_header=True,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": USER_AGENT})
    return s


def preview_text(raw, n=200):
    """First n bytes as one printable line, for error messages."""
    return " ".join(raw[:n].decode("utf-8", errors="replace").split())


def probe_url(session, url, allowed_hosts, timeout=(10, 30), expect_json=False, params=None):
    """Tiny request: can this compute reach the host, does the URL exist, and (if expect_json) is the body real data?
    A 200 is not proof by itself: some APIs answer an error with HTTP 200 and a text or HTML body."""
    host = urlparse(url).hostname
    headers = {} if expect_json else {"Range": "bytes=0-1023"}
    try:
        r = session.get(url, params=params, headers=headers, stream=True, timeout=timeout)
        code, final = r.status_code, r.url
        body = r.content[:65536] if expect_json else b""
        r.close()
    except requests.exceptions.RequestException as e:
        # exception text often embeds the full URL, key included
        return dict(host=host, reachable=False, http_status=None, disallowed_redirect=False,
                    unexpected_content=False, error=scrub(f"{type(e).__name__}: {str(e)[:200]}"))
    if not host_allowed(urlparse(final).hostname, allowed_hosts):
        return dict(host=host, reachable=False, http_status=code, disallowed_redirect=True,
                    unexpected_content=False, error=f"redirected to disallowed host {urlparse(final).hostname}")
    reachable = code in (200, 206)
    if reachable and expect_json:
        try:
            data = json.loads(body.decode("utf-8"))
            if not (isinstance(data, list) and len(data) >= 2):
                raise ValueError("not a header+rows array")
        except ValueError:   # JSONDecodeError and UnicodeDecodeError both subclass ValueError
            return dict(host=host, reachable=False, http_status=code, disallowed_redirect=False,
                        unexpected_content=True,
                        error=scrub(f"HTTP {code} but the body is not the expected JSON; it starts: {preview_text(body)!r}"))
    return dict(host=host, reachable=reachable, http_status=code, disallowed_redirect=False,
                unexpected_content=False, error=None)


def probe_to_status(p):
    """200 with a wrong body = unexpected_content; 404 = does not exist; no response / 401 / 403 / 407 / bad redirect = blocked; else unavailable."""
    if p["reachable"]:
        return "ok"
    if p.get("unexpected_content"):
        return "unexpected_content"
    if p["http_status"] == 404:
        return "not_found"
    if p["http_status"] is None or p["http_status"] in (401, 403, 407) or p["disallowed_redirect"]:
        return "blocked"
    return "unavailable"


def candidate_years(src, now=None):
    current = (now or datetime.now(timezone.utc)).year
    yc = src["year_candidates"]
    return [current - yc["start_offset"] - i for i in range(yc["count"])]


def resolve_source(session, src, now=None):
    """Turn a registry entry into a concrete URL + vintage, or an explicit non-ok status."""
    base = dict(source=src["name"], strategy=src["strategy"], url=None, vintage=None,
                status="ok", http_status=None, detail=None)
    strat = src["strategy"]

    if strat == "manual":
        return {**base, "status": "skipped_manual", "detail": " ".join(str(src.get("manual_note", "")).split())}

    hosts = src["allowed_hosts"]

    cfg = src.get("api_key")
    key_params = key_params_for(src)
    if cfg and cfg.get("required") and not key_params:
        where = f"secret {cfg['secret_scope']}/{cfg['secret_key']} or " if cfg.get("secret_scope") else ""
        return {**base, "status": "needs_key",
                "detail": f"this source requires an API key: set {where}the '{cfg['widget']}' widget"}

    if strat == "static":
        p = probe_url(session, src["url"], hosts, params=key_params)
        status = probe_to_status(p)
        if status == "ok":
            return {**base, "url": src["url"], "vintage": src.get("vintage_label"), "http_status": p["http_status"]}
        return {**base, "status": status, "http_status": p["http_status"], "detail": p["error"] or f"HTTP {p['http_status']}"}

    # latest_year: newest first. 404 = not published yet, try the previous year. Anything else stops the search.
    years = [int(src["pin"])] if src.get("pin") else candidate_years(src, now)
    tried = []
    for y in years:
        p = probe_url(session, format_url(src["probe_template"], src, y), hosts,
                      expect_json=(src.get("probe_expect") == "json"), params=key_params)
        tried.append((y, p["http_status"]))
        status = probe_to_status(p)
        if status == "ok":
            return {**base, "url": format_url(src["url_template"], src, y),
                    "vintage": src["vintage_label"].format(year=y), "http_status": p["http_status"],
                    "detail": f"resolved year {y}; tried {tried}"}
        if status == "not_found":
            continue
        return {**base, "status": status, "http_status": p["http_status"],
                "detail": f"year {y}: {p['error'] or 'HTTP ' + str(p['http_status'])}; tried {tried}"}
    return {**base, "status": "not_found", "detail": f"no candidate vintage found; tried {tried}"}


def fetch_to_local(session, url, tmp_dir, allowed_hosts, timeout=(10, 120), params=None):
    """Stream to local disk while hashing. Rejects a redirect to a host outside the allow-list."""
    h, n = hashlib.sha256(), 0
    local = os.path.join(tmp_dir, uuid.uuid4().hex)
    with session.get(url, params=params, stream=True, timeout=timeout) as r:
        final_host = urlparse(r.url).hostname
        if not host_allowed(final_host, allowed_hosts):
            raise ValueError(f"redirected to disallowed host {final_host}")
        ctype = r.headers.get("content-type")
        if r.status_code != 200:
            return r.status_code, None, 0, None, ctype
        with open(local, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                h.update(chunk)
                n += len(chunk)
        return r.status_code, local, n, h.hexdigest(), ctype


def land_file(local_path, subdir, stem, ext):
    """Copy into the Volume under a dated name. Never overwrites an existing file."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d")
    dest_dir = f"{VOLUME_ROOT}/{subdir}"
    os.makedirs(dest_dir, exist_ok=True)
    dest = f"{dest_dir}/{stem}_{stamp}{ext}"
    if os.path.exists(dest):
        dest = f"{dest_dir}/{stem}_{stamp}_{uuid.uuid4().hex[:6]}{ext}"
    shutil.copyfile(local_path, dest)
    return dest
# --- CORE END ---

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Manifest table
# MAGIC One row per attempt. `status`: `landed`, `unchanged` (same hash as a previous landing, nothing written), `skipped_manual`, `needs_key` (source requires an API key and none was provided), `blocked` (host unreachable or 401/403/bad redirect), `not_found` (404), `unexpected_content` (HTTP 200 but not the data, e.g. an error page), `unavailable` (5xx/429 after retries), `failed_fetch`, `failed_validation`.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType, TimestampType

MANIFEST_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("fetched_at", TimestampType()),
    StructField("source", StringType()),
    StructField("method", StringType()),        # http | manual_upload
    StructField("url", StringType()),
    StructField("http_status", IntegerType()),
    StructField("status", StringType()),
    StructField("bytes", LongType()),
    StructField("sha256", StringType()),
    StructField("landed_path", StringType()),
    StructField("detail", StringType()),
    StructField("vintage", StringType()),       # e.g. "ACS 5-year ending 2024", "ZORI as of 2026-07-31"
])

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (
        run_id STRING, fetched_at TIMESTAMP, source STRING, method STRING, url STRING,
        http_status INT, status STRING, bytes BIGINT, sha256 STRING, landed_path STRING,
        detail STRING, vintage STRING
    ) COMMENT 'One row per ingestion attempt (including failures and blocked hosts). Bronze provenance.'
""")
if "vintage" not in spark.table(MANIFEST_TABLE).columns:    # table created by an earlier version of this notebook
    spark.sql(f"ALTER TABLE {MANIFEST_TABLE} ADD COLUMNS (vintage STRING)")

RUN_ID = uuid.uuid4().hex[:12]


def record(source, method, url, status, http_status=None, nbytes=None, sha=None,
           path=None, detail=None, vintage=None):
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    # last line of defence: nothing registered as a secret is ever written to the manifest
    row = (RUN_ID, now, source, method, scrub(url), http_status, status, nbytes, sha, path, scrub(detail), vintage)
    spark.createDataFrame([row], MANIFEST_SCHEMA).write.mode("append").saveAsTable(MANIFEST_TABLE)


def already_landed(source, sha):
    return (
        spark.table(MANIFEST_TABLE)
        .filter((F.col("source") == source) & (F.col("sha256") == sha)
                & F.col("status").isin("landed", "unchanged"))
        .limit(1).count() > 0
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Load and validate the registry
# MAGIC A typo in `sources.yml` stops the run here with every problem listed.

# COMMAND ----------

CONFIG_PATH = get_param("config_path", "config/sources.yml")
SELECTED = get_param("sources", "all")

ALL_SOURCES = load_registry(CONFIG_PATH)
validate_registry(ALL_SOURCES)

wanted = None if SELECTED.lower() == "all" else {x.strip() for x in SELECTED.split(",")}
unknown = (wanted or set()) - {s["name"] for s in ALL_SOURCES}
assert not unknown, f"Unknown source name(s) in 'sources' widget: {sorted(unknown)}"

SOURCES = [s for s in ALL_SOURCES if s.get("enabled", True) and (wanted is None or s["name"] in wanted)]
print(f"Registry OK: {len(ALL_SOURCES)} defined, {len(SOURCES)} selected -> {[s['name'] for s in SOURCES]}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Resolve every source from this compute
# MAGIC Read this table before anything downloads. `blocked` means the host is outside this workspace's allowed domains (or the URL redirected somewhere not on the allow-list). `not_found` on a `static` source means the registry URL is wrong or has moved.

# COMMAND ----------

session = make_session()
resolutions = [resolve_source(session, s) for s in SOURCES]
display(spark.createDataFrame(
    [(r["source"], r["strategy"], r["status"], r["vintage"], r["http_status"], scrub(r["url"]), scrub(r["detail"])) for r in resolutions],
    "source string, strategy string, status string, vintage string, http_status int, url string, detail string",
))

blocked = [r["source"] for r in resolutions if r["status"] == "blocked"]
if blocked:
    print("BLOCKED from this compute:", blocked, "-> use the manual fallback in section 5 for these.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Download, validate, land, log

# COMMAND ----------

by_name = {r["source"]: r for r in resolutions}
outcomes = []

with tempfile.TemporaryDirectory() as tmp_dir:
    for src in SOURCES:
        name = src["name"]
        res = by_name[name]

        if res["status"] != "ok":
            record(name, "http", res["url"], res["status"], http_status=res["http_status"], detail=res["detail"])
            outcomes.append((name, res["status"], res["detail"]))
            continue

        url = res["url"]
        try:
            code, local, n, sha, ctype = fetch_to_local(
                session, url, tmp_dir, src["allowed_hosts"], params=key_params_for(src))
        except (requests.exceptions.RequestException, ValueError) as e:
            detail = scrub(f"{type(e).__name__}: {str(e)[:200]}")
            record(name, "http", url, "failed_fetch", vintage=res["vintage"], detail=detail)
            outcomes.append((name, "failed_fetch", detail))
            continue

        if local is None:
            detail = f"HTTP {code}"
            record(name, "http", url, "failed_fetch", http_status=code, vintage=res["vintage"], detail=detail)
            outcomes.append((name, "failed_fetch", detail))
            continue

        if n == 0:   # a 200 with an empty body would otherwise land as a 0-byte "file"
            detail = f"empty response body (content-type={ctype})"
            record(name, "http", url, "failed_validation", http_status=code, nbytes=0, sha=sha,
                   vintage=res["vintage"], detail=detail)
            outcomes.append((name, "failed_validation", detail))
            continue

        try:
            ok, detail, file_vintage = VALIDATORS[src["validator"]](local, src)
        except Exception as e:   # unparseable content (e.g. an HTML error page) counts as a validation failure
            ok, detail, file_vintage = False, f"{type(e).__name__}: {str(e)[:200]}", None
        vintage = file_vintage or res["vintage"]

        if not ok:
            # Keep the evidence: the rejected file is discarded, so record what it actually was.
            with open(local, "rb") as fh:
                head = preview_text(fh.read(300))
            detail = f"{detail} | content-type={ctype} | body starts: {head!r}"
            record(name, "http", url, "failed_validation", http_status=code, nbytes=n, sha=sha,
                   vintage=vintage, detail=detail)
            outcomes.append((name, "failed_validation", detail))
            continue

        if already_landed(name, sha):
            record(name, "http", url, "unchanged", http_status=code, nbytes=n, sha=sha, vintage=vintage,
                   detail=f"identical to a previous landing; {detail}")
            outcomes.append((name, "unchanged", f"{vintage}: {detail}"))
            continue

        dest = land_file(local, src["subdir"], src["stem"], src["ext"])
        record(name, "http", url, "landed", http_status=code, nbytes=n, sha=sha, path=dest,
               vintage=vintage, detail=detail)
        outcomes.append((name, "landed", f"{vintage}: {detail} -> {dest}"))

# Explicit schema: Spark cannot infer a type for a column that is None in every row.
display(spark.createDataFrame(outcomes, "source string, status string, detail string"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Manual fallback: register a file you uploaded by hand
# MAGIC For sources that came back `blocked` or `skipped_manual`: upload into the matching Volume folder (`fips_reference/`, `zori/`, `acs/`, `chas/`), then register it so it gets the same hash, validation and manifest record as a downloaded file.

# COMMAND ----------

def register_manual_file(source_name, volume_path):
    src = next(s for s in ALL_SOURCES if s["name"] == source_name)
    h, n = hashlib.sha256(), 0
    with open(volume_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
            n += len(chunk)
    sha = h.hexdigest()

    if n == 0:
        ok, detail, file_vintage = False, "empty file", None
    elif src.get("validator"):
        try:
            ok, detail, file_vintage = VALIDATORS[src["validator"]](volume_path, src)
        except Exception as e:
            ok, detail, file_vintage = False, f"{type(e).__name__}: {str(e)[:200]}", None
    else:
        ok, detail, file_vintage = True, "no validator configured for this source", None

    label = src.get("vintage_label")
    vintage = file_vintage or (label if label and "{year}" not in label else None)
    status = "landed" if ok else "failed_validation"
    record(source_name, "manual_upload", None, status, nbytes=n, sha=sha, path=volume_path,
           detail=detail, vintage=vintage)
    print(f"{source_name}: {status} - {detail}")

# Examples (uncomment and edit):
# register_manual_file("census_county_reference", f"{VOLUME_ROOT}/fips_reference/canonical_national_county2020.txt")
# register_manual_file("hud_chas_county", f"{VOLUME_ROOT}/chas/<your_file>.zip")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Fail loudly if anything needs attention
# MAGIC `landed`, `unchanged` and `skipped_manual` are fine. Anything else stops the run, so a scheduled job can never look green while a source silently went stale.

# COMMAND ----------

display(
    spark.table(MANIFEST_TABLE)
    .filter(F.col("run_id") == RUN_ID)
    .select("source", "method", "status", "vintage", "http_status", "bytes", "landed_path", "detail")
)

needs_attention = [(n, s, d) for n, s, d in outcomes if s not in ("landed", "unchanged", "skipped_manual")]
if needs_attention:
    lines = "\n".join(f"  - {n}: {s} ({d})" for n, s, d in needs_attention)
    raise RuntimeError(f"{len(needs_attention)} source(s) need attention:\n{lines}")
print("Ingestion finished with no failures.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Notes for silver
# MAGIC - **ACS sentinel values:** the Census API returns large negative numbers (e.g. `-666666666`) where an estimate or margin is unavailable. Bronze keeps them raw. In silver, convert to NULL **and** keep the reason in a separate field, otherwise a fake rent of -$666 million lands in your averages.
# MAGIC - **Read every FIPS column as a string.** Zillow's raw file has zero-padded codes (`01`, `073`), but a default pandas `read_csv` turns them into integers (`1`, `73`) and the geoid becomes `173` instead of `01073`. In Spark, declare the schema explicitly, as in `01_build_fips_spine`.
# MAGIC - **Zillow's county file is Zillow's own county rollup**, not a rollup of ZIP rows you computed. Record which method any coverage number came from.
# MAGIC - **Connecticut:** Zillow's file uses the 8 legacy counties (`09001`-`09015`). After the first ACS landing, check which Connecticut GEOIDs ACS uses.